In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# List of (month label, file path) tuples
month_files = [
    ('April 2024', 'All Calls by Month/April 2024.xlsx'),
    ('August 2024', 'All Calls by Month/August 2024.xlsx'),
    ('December 2024', 'All Calls by Month/December 2024.xlsx'),
    ('February 2025', 'All Calls by Month/February 2025.xlsx'),
    ('January 2025', 'All Calls by Month/January 2025.xlsx'),
    ('July 2024', 'All Calls by Month/July 2024.xlsx'),
    ('June 2024', 'All Calls by Month/June 2024.xlsx'),
    ('March 2025', 'All Calls by Month/March 2025.xlsx'),
    ('May 2024', 'All Calls by Month/May 2024.xlsx'),
    ('October 2024', 'All Calls by Month/October 2024.xlsx'),
    ('November 2024', 'All Calls by Month/November 2024.xlsx'),
    ('September 2024', 'All Calls by Month/September 2024.xlsx'),
]

# Read and combine all DataFrames
dfs = []

for month, path in month_files:
    df = pd.read_excel(path)
    df['Month'] = month

    # moving month column to first column
    cols = df.columns.tolist()
    cols.insert(0, cols.pop(cols.index('Month')))
    df = df[cols]
    
    dfs.append(df)

# Concatenate all into a single DataFrame
all_calls = pd.concat(dfs, ignore_index = True)

print(all_calls.shape)
print(all_calls.columns.tolist())

# Convert datetime columns
datetime_columns = ['Start time', 'Answer time', 'Release time', 'Report time']

for col in datetime_columns:
    if col in all_calls.columns:
        all_calls[col] = pd.to_datetime(all_calls[col], errors='coerce')

# Quick check
all_calls.head()

# Adding New Columns
all_calls['Hour'] = all_calls['Start time'].dt.hour
    # what hour did they call based on start time
all_calls['DayOfWeek'] = all_calls['Start time'].dt.day_name()
    # what day of the week did they call based on start time

all_calls['Date'] = all_calls['Start time'].dt.date


print(all_calls.columns)
print(all_calls.shape)

In [ ]:
columns_to_keep = ['Month','Correlation ID', 'Direction', 'PSTN legal entity',
                   'Duration', 'Called number', 'Hour','DayOfWeek', 'Date',
                   'PSTN vendor Org ID', 'PSTN vendor name', 'PSTN provider ID',
                   'Month', 'Start time', 'Answer time', 'Call type', 'Answered',
                   'Call outcome'
                   ]

df_filtered = all_calls[columns_to_keep]

df = pd.DataFrame(df_filtered)

df.to_csv('all_calls_concat_filtered.csv', index=False)
    # this is the data that is used within the dashboard
    # other edits to the data are created as calculated fields (under analysis tab) within Tableau.

In [ ]:
import tkinter as tk
from tkinter import filedialog

# Launch a file dialog to select the Excel file
root = tk.Tk()
root.withdraw()  # ?? hide the root window
file_path = filedialog.askopenfilename(title="Select Excel file", filetypes=[("Excel files", "*.xlsx *.xls")])

if file_path:
    try:
        # Read the Excel file
        df = pd.read_excel(file_path)

        # List of relevant columns to select
        columns_to_keep = [
            'Correlation ID', 'Direction', 'PSTN legal entity',
            'PSTN vendor Org ID', 'PSTN vendor name', 'PSTN provider ID',
            'Month', 'Start time', 'Answer time', 'Duration', 'Called number'
        ]

        # Select the relevant columns (if they exist)
        df_filtered = df[columns_to_keep]

        # Save the filtered DataFrame to CSV
        output_path = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv")],
            title="Save filtered CSV file as"
        )
        if output_path:
            df_filtered.to_csv(output_path, index=False)
            print(f"Filtered CSV saved to: {output_path}")
        else:
            print("Save operation cancelled.")

    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print("File selection cancelled.")

# user will then upload csv file to Tableau which will then analyze it with the All Calls dashboard as Sarah and Sadie have created. 
